# CATNAT Seismic Resilience Platform - Phase II
## Advanced Portfolio Analysis & Dashboard Foundation (Weeks 5–8)

**Objective:**
- 3D portfolio segmentation (TYPE × ZONE × WILAYA)
- Advanced concentration metrics
- Stress testing & capital reduction scenarios
- Streamlit dashboard prototype
- REST API expansion

**Inputs:** portfolio_enriched.parquet, pml_scenarios.csv  
**Outputs:** dashboard_data.parquet, api_ready_data.json, streamlit_app.py

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# Configuration
DATA_DIR = Path(r"c:\Users\WINDOWS\OneDrive\Desktop\Sys\data")
INPUT_PARQUET = DATA_DIR / "portfolio_enriched.parquet"
PML_FILE = DATA_DIR / "pml_scenarios.csv"

print("=" * 80)
print("PHASE II: ADVANCED PORTFOLIO ANALYSIS & DASHBOARD")
print("=" * 80)

# Load data
print("\n📥 Loading datasets...")
df = pd.read_parquet(INPUT_PARQUET)
pml_df = pd.read_csv(PML_FILE, decimal=',')
print(f"✓ Portfolio: {len(df):,} policies | ${df['CAPITAL_ASSURE'].sum():,.0f} capital")
print(f"✓ PML scenarios: {len(pml_df)} scenarios loaded")

PHASE II: ADVANCED PORTFOLIO ANALYSIS & DASHBOARD

📥 Loading datasets...
✓ Portfolio: 39,196 policies | $299,542,211,932 capital
✓ PML scenarios: 3 scenarios loaded


In [2]:
# 1. 3D PORTFOLIO SEGMENTATION

print("\n" + "=" * 80)
print("SECTION 1: 3D PORTFOLIO SEGMENTATION (TYPE × ZONE × WILAYA)")
print("=" * 80)

# Create 3D pivot analysis
pivot_3d = df.groupby(['TYPE', 'ZONE_SISMIQUE', 'WILAYA']).agg({
    'NUMERO_POLICE': 'count',
    'CAPITAL_ASSURE': 'sum',
    'VULNERABILITY_SCORE': 'mean',
    'PRIME_NETTE': 'sum'
}).reset_index()

pivot_3d.columns = ['Type', 'Zone', 'Wilaya', 'Policies', 'Capital', 'Avg_Vulnerability', 'Premium']
pivot_3d = pivot_3d.sort_values('Capital', ascending=False)

print(f"\n✓ Created 3D segmentation: {len(pivot_3d)} unique combinations")

# Top combinations
print("\nTOP 30 TYPE×ZONE×WILAYA COMBINATIONS BY CAPITAL:")
top_30 = pivot_3d.head(30)[['Type', 'Zone', 'Wilaya', 'Policies', 'Capital', 'Avg_Vulnerability']]
print(top_30.to_string(index=False))

# Export
pivot_3d.to_csv(DATA_DIR / 'portfolio_3d_segmentation.csv', index=False, decimal=',')
print(f"\n✓ Exported to: portfolio_3d_segmentation.csv")


SECTION 1: 3D PORTFOLIO SEGMENTATION (TYPE × ZONE × WILAYA)

✓ Created 3D segmentation: 124 unique combinations

TOP 30 TYPE×ZONE×WILAYA COMBINATIONS BY CAPITAL:
                         Type Zone            Wilaya  Policies      Capital  Avg_Vulnerability
1 - Installation Industrielle  IIa        16 - ALGER        45 4.167009e+10               0.48
              Bien immobilier  IIa        16 - ALGER      5547 2.970097e+10               0.30
 2 - Installation Commerciale  IIa        16 - ALGER      3133 2.905349e+10               0.36
1 - Installation Industrielle  IIb        19 - SETIF        55 1.952084e+10               0.64
              Bien immobilier  IIa   15 - TIZI OUZOU      3782 1.630241e+10               0.30
1 - Installation Industrielle  IIa         31 - ORAN        11 1.580811e+10               0.48
              Bien immobilier  IIb        19 - SETIF      3690 1.125160e+10               0.40
 2 - Installation Commerciale  IIa         31 - ORAN       297 9.804224e+09  

In [3]:
# 2. STRESS TESTING & CAPITAL REDUCTION SCENARIOS

print("\n" + "=" * 80)
print("SECTION 2: STRESS TESTING & CAPITAL REDUCTION SCENARIOS")
print("=" * 80)

total_capital = df['CAPITAL_ASSURE'].sum()
zone_iii_capital = df[df['ZONE_SISMIQUE'] == 'III']['CAPITAL_ASSURE'].sum()

print(f"\n📊 BASELINE (Current State):")
print(f"  - Total Capital: ${total_capital:,.0f}")
print(f"  - Zone III Capital: ${zone_iii_capital:,.0f} ({zone_iii_capital/total_capital*100:.1f}%)")
print(f"  - Exposure Level: {'CRITICAL (>15%)' if zone_iii_capital/total_capital > 0.15 else 'HIGH (10-15%)' if zone_iii_capital/total_capital > 0.10 else 'MODERATE'}")

# Define stress scenarios
stress_scenarios = []

for reduction_target in [10, 15, 20, 25, 30]:
    # Scenario: Reduce Zone III by X%
    zone_iii_df = df[df['ZONE_SISMIQUE'] == 'III'].copy()
    reduction_amount = zone_iii_capital * (reduction_target / 100)
    
    # Identify policies to reduce (by highest vulnerability first)
    zone_iii_sorted = zone_iii_df.sort_values('VULNERABILITY_SCORE', ascending=False)
    cumsum = zone_iii_sorted['CAPITAL_ASSURE'].cumsum()
    policies_to_reduce = zone_iii_sorted[cumsum <= reduction_amount]
    
    new_capital = total_capital - reduction_amount
    new_zone_iii_pct = (zone_iii_capital - reduction_amount) / new_capital * 100
    
    stress_scenarios.append({
        'Scenario': f'Zone III -{reduction_target}%',
        'Reduction_Target': f'{reduction_target}%',
        'Capital_Reduction': reduction_amount,
        'Policies_Affected': len(policies_to_reduce),
        'New_Total_Capital': new_capital,
        'New_Zone_III_Capital': zone_iii_capital - reduction_amount,
        'New_Zone_III_Pct': new_zone_iii_pct,
        'Status': 'Target Met' if new_zone_iii_pct < 10 else 'In Progress'
    })

stress_df = pd.DataFrame(stress_scenarios)

print(f"\n📈 STRESS TEST SCENARIOS:")
print(stress_df[['Scenario', 'Policies_Affected', 'New_Zone_III_Pct', 'Status']].to_string(index=False))

# Export
stress_df.to_csv(DATA_DIR / 'stress_test_scenarios.csv', index=False, decimal=',')
print(f"\n✓ Exported to: stress_test_scenarios.csv")


SECTION 2: STRESS TESTING & CAPITAL REDUCTION SCENARIOS

📊 BASELINE (Current State):
  - Total Capital: $299,542,211,932
  - Zone III Capital: $19,880,937,758 (6.6%)
  - Exposure Level: MODERATE

📈 STRESS TEST SCENARIOS:
     Scenario  Policies_Affected  New_Zone_III_Pct     Status
Zone III -10%                 27          6.013307 Target Met
Zone III -15%                 68          5.698271 Target Met
Zone III -20%                 68          5.381116 Target Met
Zone III -25%                 71          5.061820 Target Met
Zone III -30%                 71          4.740362 Target Met

✓ Exported to: stress_test_scenarios.csv


In [4]:
# 3. DASHBOARD DATA PREPARATION

print("\n" + "=" * 80)
print("SECTION 3: DASHBOARD DATA PREPARATION")
print("=" * 80)

# Prepare data for Streamlit dashboard
dashboard_data = {
    'summary': {
        'total_policies': int(len(df)),
        'total_capital': float(df['CAPITAL_ASSURE'].sum()),
        'total_premium': float(df['PRIME_NETTE'].sum()),
        'active_policies': int((df['POLICY_STATUS'] == 'ACTIVE').sum()),
        'data_quality': 94.4
    },
    
    'by_zone': df.groupby('ZONE_SISMIQUE').agg({
        'NUMERO_POLICE': 'count',
        'CAPITAL_ASSURE': 'sum'
    }).to_dict('index'),
    
    'by_type': df.groupby('TYPE').agg({
        'NUMERO_POLICE': 'count',
        'CAPITAL_ASSURE': 'sum'
    }).to_dict('index'),
    
    'top_wilayas': df.groupby('WILAYA')['CAPITAL_ASSURE'].sum().nlargest(15).to_dict(),
    
    'top_communes': df.groupby('COMMUNE')['CAPITAL_ASSURE'].sum().nlargest(20).to_dict(),
    
    'pml_scenarios': pml_df.to_dict('records')
}

# Convert numpy types for JSON serialization
def convert_types(obj):
    if isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    elif isinstance(obj, dict):
        return {k: convert_types(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_types(item) for item in obj]
    return obj

dashboard_data_clean = convert_types(dashboard_data)

# Export as both JSON and Parquet
json_path = DATA_DIR / 'dashboard_data.json'
with open(json_path, 'w') as f:
    json.dump(dashboard_data_clean, f, indent=2)

df.to_parquet(DATA_DIR / 'dashboard_data.parquet')
print(f"✓ Dashboard data exported to:")
print(f"  - {json_path}")
print(f"  - {DATA_DIR / 'dashboard_data.parquet'}")


SECTION 3: DASHBOARD DATA PREPARATION
✓ Dashboard data exported to:
  - c:\Users\WINDOWS\OneDrive\Desktop\Sys\data\dashboard_data.json
  - c:\Users\WINDOWS\OneDrive\Desktop\Sys\data\dashboard_data.parquet


In [5]:
# 4. ADVANCED CONCENTRATION METRICS

print("\n" + "=" * 80)
print("SECTION 4: ADVANCED CONCENTRATION METRICS")
print("=" * 80)

# Herfindahl Index (concentration measure)
total_capital = df['CAPITAL_ASSURE'].sum()
zone_shares = (df.groupby('ZONE_SISMIQUE')['CAPITAL_ASSURE'].sum() / total_capital) ** 2
herfindahl_zone = zone_shares.sum()

wilaya_shares = (df.groupby('WILAYA')['CAPITAL_ASSURE'].sum() / total_capital) ** 2
herfindahl_wilaya = wilaya_shares.sum()

print(f"\n📊 HERFINDAHL CONCENTRATION INDEX (0=perfect diversity, 1=monolithic):")
print(f"  By Zone:   {herfindahl_zone:.4f}")
print(f"  By Wilaya: {herfindahl_wilaya:.4f}")

# Gini Coefficient (wealth inequality)
def gini(x):
    sorted_x = np.sort(x)
    n = len(x)
    cumsum = np.cumsum(sorted_x)
    return (2 * np.sum(cumsum)) / (n * cumsum[-1]) - (n + 1) / n

gini_coeff = gini(df['CAPITAL_ASSURE'].values)
print(f"\n📊 GINI COEFFICIENT (inequality measure):")
print(f"  Capital Distribution Gini: {gini_coeff:.3f}")
print(f"  Interpretation: {'Highly concentrated' if gini_coeff > 0.7 else 'Moderately concentrated' if gini_coeff > 0.5 else 'Well distributed'}")

# Concentration ratios
print(f"\n📊 CAPITAL CONCENTRATION RATIOS:")
top_10_pct = df.nlargest(int(len(df)*0.1), 'CAPITAL_ASSURE')['CAPITAL_ASSURE'].sum()
top_20_pct = df.nlargest(int(len(df)*0.2), 'CAPITAL_ASSURE')['CAPITAL_ASSURE'].sum()
print(f"  Top 10% of policies: {top_10_pct/total_capital*100:.1f}% of capital")
print(f"  Top 20% of policies: {top_20_pct/total_capital*100:.1f}% of capital")

print(f"\n✅ Phase II Analysis Complete")


SECTION 4: ADVANCED CONCENTRATION METRICS

📊 HERFINDAHL CONCENTRATION INDEX (0=perfect diversity, 1=monolithic):
  By Zone:   0.3841
  By Wilaya: 0.1515

📊 GINI COEFFICIENT (inequality measure):
  Capital Distribution Gini: nan
  Interpretation: Well distributed

📊 CAPITAL CONCENTRATION RATIOS:
  Top 10% of policies: 70.1% of capital
  Top 20% of policies: 77.4% of capital

✅ Phase II Analysis Complete
